In [25]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_18'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Working directory: /content/BITS_programming/assignments/assignment_18


# 3. Lesson 3 Hands-On Lab — Advanced Pipeline Orchestration

This notebook is intentionally simple and executable.

It uses AWS services through `boto3` only — no Kubeflow, no Ray install, no simulation classes.

You will build a production-grade MLOps workflow covering:
- **Multi-environment pipeline** — dev → staging → prod using S3 prefixes and promotion logic
- **SageMaker Experiments** — cross-run comparison of v1 baseline vs v2 retrained model
- **KS-test drift detection** — statistical comparison of baseline vs live inference distribution
- **Automated retraining trigger** — CloudWatch alarm fires when drift exceeds threshold
- **Model versioning** — v1 and v2 registered in SageMaker Model Registry, same group
- **Automated rollback** — if v2 fails the accuracy gate, v1 is retrieved from registry automatically


## 3.1 Environment Setup

### 3.1.1 Import Libraries

In [26]:
# Block 1 - Import libraries
!pip install boto3
from google.colab import userdata

def get_colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value

import json
import hashlib
import datetime as dt
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd

from botocore.exceptions import ClientError
from scipy.stats import ks_2samp
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")

Libraries imported successfully


### 3.1.2 Connect to AWS

In [27]:
# Fetch AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = get_colab_secret('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = get_colab_secret('AWS_SECRET_ACCESS_KEY')
AWS_SESSION_TOKEN = get_colab_secret('AWS_SESSION_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

print("AWS credentials loaded from Colab secrets and set as environment variables.")

# Block 2 - Create AWS clients

session = boto3.Session()
region  = session.region_name or "us-east-2"

s3               = session.client("s3",         region_name=region)
sts              = session.client("sts",         region_name=region)
cloudwatch       = session.client("cloudwatch",  region_name=region)
sagemaker_client = session.client("sagemaker",   region_name=region)

identity   = sts.get_caller_identity()
account_id = identity["Account"]

print("Connected to AWS")
print("Region  :", region)
print("Account :", account_id)
print("Caller  :", identity["Arn"])

AWS credentials loaded from Colab secrets and set as environment variables.
Connected to AWS
Region  : us-east-2
Account : 455865672536
Caller  : arn:aws:iam::455865672536:root


### 3.1.3 Configure Multi-Environment Paths

In production, each environment has its own S3 prefix, IAM boundary, and model registry approval tier.
Here all three environments live as distinct S3 prefixes — the same pattern used by enterprise MLOps platforms.


In [28]:
# Block 3 - Configure multi-environment S3 paths

project_name     = "lesson3-advanced-pipeline"
run_id           = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
model_group_name = f"lesson3-loan-risk-{account_id}"
experiment_name  = "lesson3-advanced-pipeline-exp"
namespace        = "Lesson3/AdvancedPipeline"

bucket_name = f"{project_name}-{account_id}-{region}".replace("_", "-").lower()

ENVS = {"dev": "lesson3/dev", "staging": "lesson3/staging", "prod": "lesson3/prod"}

def env_paths(env):
    p = ENVS[env]
    return {
        "raw":      f"{p}/data/raw/loan_data.csv",
        "train":    f"{p}/data/processed/train.csv",
        "test":     f"{p}/data/processed/test.csv",
        "model":    f"{p}/model/model_{run_id}.joblib",
        "baseline": f"{p}/monitoring/baseline.json",
        "drift":    f"{p}/monitoring/drift_report_{run_id}.json",
    }

paths = {env: env_paths(env) for env in ENVS}

local_dir = Path("lesson3_outputs")
local_dir.mkdir(exist_ok=True)

print("Bucket      :", bucket_name)
print("Experiment  :", experiment_name)
print("Model Group :", model_group_name)
print("Run ID      :", run_id)
print()
for env in ENVS:
    print(f"  [{env}] prefix: {ENVS[env]}/")

Bucket      : lesson3-advanced-pipeline-455865672536-us-east-2
Experiment  : lesson3-advanced-pipeline-exp
Model Group : lesson3-loan-risk-455865672536
Run ID      : 20260915-060957

  [dev] prefix: lesson3/dev/
  [staging] prefix: lesson3/staging/
  [prod] prefix: lesson3/prod/


/tmp/ipykernel_2187/3897064964.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id           = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")


## 3.2 S3 Multi-Environment Setup

### 3.2.1 Create or Reuse S3 Bucket

In [30]:
try:
    s3.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={"LocationConstraint": region}
    )
    print(f"Created S3 bucket: {bucket_name}")
except ClientError as e:
    if e.response["Error"]["Code"] == "BucketAlreadyOwnedByYou":
        print(f"Bucket {bucket_name} already exists and is owned by you. Reusing it.")
    else:
        raise

Bucket lesson3-advanced-pipeline-455865672536-us-east-2 already exists and is owned by you. Reusing it.


### 3.2.2 Generate Loan Risk Dataset and Upload to Dev

In [31]:
# Block 5 - Generate dataset and upload to dev S3 prefix

columns = [
    "income_score", "credit_history_score", "debt_ratio_score",
    "employment_score", "savings_score", "repayment_behavior_score"
]

X, y = make_classification(
    n_samples=1500, n_features=6, n_informative=4,
    n_redundant=1, n_classes=2, random_state=42
)
df = pd.DataFrame(X, columns=columns)
df["loan_default_risk"] = y

raw_path = local_dir / "loan_data.csv"
df.to_csv(raw_path, index=False)
s3.upload_file(str(raw_path), bucket_name, paths["dev"]["raw"])

display(df.head())
print("Rows     :", len(df))
print("Uploaded :", f"s3://{bucket_name}/{paths['dev']['raw']}")

,income_score,credit_history_score,debt_ratio_score,employment_score,savings_score,repayment_behavior_score,loan_default_risk
0,1.705099,1.004242,0.588320,-0.410098,-0.282672,-2.320010,1
1,0.918796,-0.262210,1.302603,2.873578,-3.560945,-0.354622,0
2,0.526696,0.524000,-1.071553,0.197359,0.052271,-1.531038,0
3,-0.719885,0.205568,-1.516909,-0.078276,0.812511,0.107889,1
4,1.487276,-1.031392,0.447000,0.456336,-1.478771,-2.083008,1


Rows     : 1500
Uploaded : s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/dev/data/raw/loan_data.csv


## 3.3 SageMaker Experiment Setup

### 3.3.1 Create Experiment and Pipeline Helpers

In [33]:
# Block 6 - Create SageMaker Experiment + define helpers

try:
    sagemaker_client.create_experiment(
        ExperimentName=experiment_name,
        Description=(
            "Lesson 3 — Advanced Pipeline Orchestration. "
            "Tracks v1 baseline and v2 retrained runs for cross-run comparison."
        ),
        Tags=[{"Key": "Project", "Value": project_name}]
    )
    print("Created experiment:", experiment_name)
except ClientError as e:
    error_code = e.response["Error"]["Code"]
    error_message = e.response["Error"]["Message"]
    # Handle both ConflictException and ValidationException if it indicates resource already exists
    if error_code == "ConflictException" or (error_code == "ValidationException" and "already exists" in error_message):
        print("Experiment already exists — reusing:", experiment_name)
    else:
        raise


def run_stage(trial_name, stage_id, display_name, worker_fn, context,
              input_artifacts=None, param_keys=None):
    """Execute one pipeline stage as a SageMaker Trial Component.
    display_name must match: [a-zA-Z0-9](-*[a-zA-Z0-9]){0,119}
    """
    cname = f"{stage_id}-{run_id}"
    sagemaker_client.create_trial_component(
        TrialComponentName=cname,
        DisplayName=display_name,
        Status={"PrimaryStatus": "InProgress"},
        StartTime=dt.datetime.utcnow(),
        InputArtifacts={
            name: {"Value": uri, "MediaType": "text/plain"}
            for name, uri in (input_artifacts or {}).items()
        }
    )
    sagemaker_client.associate_trial_component(TrialName=trial_name, TrialComponentName=cname)
    print(f"  [{display_name}] running...")
    try:
        result = worker_fn(context) or {}
        out = {k: {"Value": v, "MediaType": "text/plain"}
               for k, v in result.items() if isinstance(v, str) and v.startswith("s3://")}
        params = {}
        for k in (param_keys or []):
            val = context.get(k)
            if isinstance(val, (int, float)):
                params[k] = {"NumberValue": float(val)}
            elif val is not None:
                params[k] = {"StringValue": str(val)}
        sagemaker_client.update_trial_component(
            TrialComponentName=cname,
            Status={"PrimaryStatus": "Completed"},
            EndTime=dt.datetime.utcnow(),
            OutputArtifacts=out,
            Parameters=params
        )
        print(f"  [{display_name}] completed")
        return result
    except Exception as exc:
        sagemaker_client.update_trial_component(
            TrialComponentName=cname, Status={"PrimaryStatus": "Failed"},
            EndTime=dt.datetime.utcnow()
        )
        print(f"  [{display_name}] FAILED: {exc}")
        raise


def split_scale_upload(env, df_raw, feat_cols):
    X = df_raw[feat_cols]
    y = df_raw["loan_default_risk"]
    sc = StandardScaler()
    Xs = pd.DataFrame(sc.fit_transform(X), columns=feat_cols)
    Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.25, random_state=42, stratify=y)
    train_df = Xtr.copy(); train_df["loan_default_risk"] = ytr.values
    test_df  = Xte.copy(); test_df["loan_default_risk"]  = yte.values
    tp = local_dir / f"train_{env}.csv"; ep = local_dir / f"test_{env}.csv"
    train_df.to_csv(tp, index=False); test_df.to_csv(ep, index=False)
    s3.upload_file(str(tp), bucket_name, paths[env]["train"])
    s3.upload_file(str(ep), bucket_name, paths[env]["test"])
    joblib.dump(sc, local_dir / f"scaler_{env}.joblib")
    return train_df, test_df

print("Helpers defined")

Experiment already exists — reusing: lesson3-advanced-pipeline-exp
Helpers defined


## 3.4 Train v1 Baseline Model

### 3.4.1 Run v1 Pipeline in Dev — Tracked in SageMaker

In [34]:
# Block 7 - Train v1 baseline model, tracked as SageMaker Trial

trial_v1 = f"v1-baseline-{run_id}"
sagemaker_client.create_trial(
    ExperimentName=experiment_name, TrialName=trial_v1,
    Tags=[{"Key": "ModelVersion", "Value": "v1"}, {"Key": "Env", "Value": "dev"}]
)
print("Trial:", trial_v1)

ctx_v1 = {}
feat_cols = columns

def w1_extract(ctx):
    ctx["row_count"] = len(pd.read_csv(local_dir / "loan_data.csv"))
    return {"raw-data": f"s3://{bucket_name}/{paths['dev']['raw']}"}

def w1_transform(ctx):
    df_raw = pd.read_csv(local_dir / "loan_data.csv")
    tr, te = split_scale_upload("dev", df_raw, feat_cols)
    ctx["train_rows"] = len(tr); ctx["test_rows"] = len(te)
    return {"train-data": f"s3://{bucket_name}/{paths['dev']['train']}",
            "test-data":  f"s3://{bucket_name}/{paths['dev']['test']}"}

def w1_train(ctx):
    tr = pd.read_csv(local_dir / "train_dev.csv")
    te = pd.read_csv(local_dir / "test_dev.csv")
    m  = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
    m.fit(tr[feat_cols], tr["loan_default_risk"])
    preds = m.predict(te[feat_cols])
    ctx["accuracy_v1"] = float(accuracy_score(te["loan_default_risk"], preds))
    mp = local_dir / "model_v1.joblib"; joblib.dump(m, mp)
    with open(mp, "rb") as f: ctx["hash_v1"] = hashlib.sha256(f.read()).hexdigest()
    s3.upload_file(str(mp), bucket_name, paths["dev"]["model"])
    print(f"    v1 accuracy: {ctx['accuracy_v1']:.4f}")
    print(classification_report(te["loan_default_risk"], preds))
    return {"model-artifact": f"s3://{bucket_name}/{paths['dev']['model']}"}

print()
run_stage(trial_v1, "v1-s1-extract",   "v1-Stage1-Extract",   w1_extract,   ctx_v1, param_keys=["row_count"])
run_stage(trial_v1, "v1-s2-transform", "v1-Stage2-Transform", w1_transform, ctx_v1,
          input_artifacts={"raw-data": f"s3://{bucket_name}/{paths['dev']['raw']}"},
          param_keys=["train_rows", "test_rows"])
run_stage(trial_v1, "v1-s3-train",     "v1-Stage3-Train",     w1_train,     ctx_v1,
          param_keys=["accuracy_v1"])

print()
print(f"v1 accuracy : {ctx_v1['accuracy_v1']:.4f}")

Trial: v1-baseline-20260915-060957



/tmp/ipykernel_2187/405646415.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=dt.datetime.utcnow(),


  [v1-Stage1-Extract] running...
  [v1-Stage1-Extract] completed


/tmp/ipykernel_2187/405646415.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=dt.datetime.utcnow(),
/tmp/ipykernel_2187/405646415.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=dt.datetime.utcnow(),


  [v1-Stage2-Transform] running...


/tmp/ipykernel_2187/405646415.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=dt.datetime.utcnow(),
/tmp/ipykernel_2187/405646415.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=dt.datetime.utcnow(),


  [v1-Stage2-Transform] completed
  [v1-Stage3-Train] running...
    v1 accuracy: 0.8560
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       188
           1       0.86      0.86      0.86       187

    accuracy                           0.86       375
   macro avg       0.86      0.86      0.86       375
weighted avg       0.86      0.86      0.86       375

  [v1-Stage3-Train] completed

v1 accuracy : 0.8560


/tmp/ipykernel_2187/405646415.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=dt.datetime.utcnow(),


### 3.4.2 Register v1 in SageMaker Model Registry

In [36]:
# Block 8 - Register v1 in SageMaker Model Registry

# Some accounts have no Model Package Group quota. Fall back to an
# unversioned Model Package so the lab can still complete its registration step.
group_exists = False
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=model_group_name,
        ModelPackageGroupDescription="Lesson 3 ? Loan risk model versions"
    )
    print("Created model package group:", model_group_name)
    group_exists = True
except ClientError as e:
    error_code = e.response["Error"]["Code"]
    error_message = e.response["Error"]["Message"]
    # Handle both ConflictException and ValidationException if it indicates resource already exists
    if error_code == "ConflictException" or (error_code == "ValidationException" and "already exists" in error_message):
        print("Reusing model package group:", model_group_name)
        group_exists = True
    elif error_code == "ResourceLimitExceeded":
        print(f"Warning: Cannot create Model Package Group due to quota limits in {region}.")
        print("Registering as an unversioned Model Package instead...")
    else:
        raise

sklearn_image_uri = (
    f"257758044811.dkr.ecr.{region}.amazonaws.com/"
    "sagemaker-scikit-learn:1.2-1-cpu-py3"
)
package_params_v1 = {
    "ModelPackageDescription": f"v1 Baseline | acc={ctx_v1['accuracy_v1']:.4f} | run={run_id}",
    "InferenceSpecification": {
        "Containers": [{
            "Image": sklearn_image_uri,
            "ModelDataUrl": f"s3://{bucket_name}/{paths['dev']['model']}",
        }],
        "SupportedContentTypes": ["text/csv", "application/json"],
        "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    },
}
if group_exists:
    package_params_v1.update({
        "ModelPackageGroupName": model_group_name,
        "ModelApprovalStatus": "Approved",
        "CustomerMetadataProperties": {
            "version": "v1", "accuracy": str(round(ctx_v1["accuracy_v1"], 4)),
            "model_artifact": f"s3://{bucket_name}/{paths['dev']['model']}",
            "model_hash": ctx_v1["hash_v1"][:32], "environment": "dev",
            "experiment": experiment_name, "trial": trial_v1, "run_id": run_id,
        },
    })
else:
    package_params_v1["ModelPackageName"] = f"{project_name}-v1-{run_id}"

resp_v1 = sagemaker_client.create_model_package(**package_params_v1)
model_package_arn_v1 = resp_v1["ModelPackageArn"]
print("v1 registered ? ARN:", model_package_arn_v1)
print("Approval status:", "Approved" if group_exists else "N/A (Unversioned Package)")

Reusing model package group: lesson3-loan-risk-455865672536
v1 registered ? ARN: arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/3
Approval status: Approved


### 3.4.3 Save Baseline Distribution to S3

In [37]:
# Block 9 - Save training feature distribution as drift baseline artifact

train_df_v1   = pd.read_csv(local_dir / "train_dev.csv")
bl_sample     = train_df_v1[feat_cols].sample(n=300, random_state=42)

baseline_artifact = {
    "created_at_utc": dt.datetime.utcnow().isoformat(),
    "model_version":  "v1",
    "feature_columns": feat_cols,
    "sample_count":   len(bl_sample),
    "statistics": {col: {"mean": float(train_df_v1[col].mean()),
                         "std":  float(train_df_v1[col].std())} for col in feat_cols},
    "samples": {col: bl_sample[col].tolist() for col in feat_cols}
}

bl_path = local_dir / "baseline.json"
bl_path.write_text(json.dumps(baseline_artifact, indent=2))
s3.upload_file(str(bl_path), bucket_name, paths["dev"]["baseline"])

print("Baseline saved — sample rows:", baseline_artifact["sample_count"])
print(f"s3://{bucket_name}/{paths['dev']['baseline']}")

/tmp/ipykernel_2187/578556741.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": dt.datetime.utcnow().isoformat(),


Baseline saved — sample rows: 300
s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/dev/monitoring/baseline.json


## 3.5 Drift Detection

### 3.5.1 Simulate Data Drift in Inference Window

In [38]:
# Block 10 - Simulate data drift — new customer population

np.random.seed(99)
X_drift, y_drift = make_classification(
    n_samples=200, n_features=6, n_informative=4, n_redundant=1, n_classes=2, random_state=99
)
df_drift = pd.DataFrame(X_drift, columns=feat_cols)
df_drift["loan_default_risk"] = y_drift

SHIFT_FEATURES = ["income_score", "debt_ratio_score", "savings_score"]
for col in SHIFT_FEATURES:
    df_drift[col] = df_drift[col] * 1.8 + 2.5

print("Drift applied to:", SHIFT_FEATURES)
print()
print(f"{'Feature':<30} {'Baseline Mean':>15} {'Drifted Mean':>14}")
print("-" * 62)
for col in feat_cols:
    bm  = train_df_v1[col].mean()
    dm  = df_drift[col].mean()
    tag = "  ⚠️" if col in SHIFT_FEATURES else ""
    print(f"  {col:<28} {bm:>15.3f} {dm:>14.3f}{tag}")

Drift applied to: ['income_score', 'debt_ratio_score', 'savings_score']

Feature                          Baseline Mean   Drifted Mean
--------------------------------------------------------------
  income_score                          -0.011          1.828  ⚠️
  credit_history_score                   0.009         -0.031
  debt_ratio_score                      -0.004          2.456  ⚠️
  employment_score                      -0.002          0.025
  savings_score                         -0.018          3.548  ⚠️
  repayment_behavior_score               0.008         -0.484


### 3.5.2 KS-Test Drift Detection

In [39]:
# Block 11 - KS-test: baseline (from S3) vs live inference window

bl_dl = local_dir / "baseline_dl.json"
s3.download_file(bucket_name, paths["dev"]["baseline"], str(bl_dl))
baseline = json.loads(bl_dl.read_text())
print(f"Baseline loaded: {baseline['sample_count']} training samples")
print()

drift_results = []
for col in feat_cols:
    ks_stat, p_value = ks_2samp(baseline["samples"][col], df_drift[col].tolist())
    drift_results.append({
        "feature":    col,
        "ks_stat":    round(float(ks_stat), 4),
        "p_value":    round(float(p_value), 4),
        "drift_flag": bool(p_value < 0.05)
    })

drift_df = pd.DataFrame(drift_results)
display(drift_df)

drifted_features = [r["feature"] for r in drift_results if r["drift_flag"]]
drift_count      = len(drifted_features)

print()
print(f"Drifted features (p < 0.05): {drifted_features}")
print(f"Total drifted              : {drift_count} / {len(feat_cols)}")

dr_path = local_dir / "drift_report.json"
dr_path.write_text(json.dumps({
    "run_id": run_id, "model_version": "v1", "drift_count": drift_count,
    "drifted_features": drifted_features, "results": drift_results,
    "timestamp": dt.datetime.utcnow().isoformat()
}, indent=2))
s3.upload_file(str(dr_path), bucket_name, paths["dev"]["drift"])

Baseline loaded: 300 training samples



,feature,ks_stat,p_value,drift_flag
0,income_score,0.4967,0.0000,True
1,credit_history_score,0.1667,0.0023,True
2,debt_ratio_score,0.6800,0.0000,True
3,employment_score,0.1750,0.0012,True
4,savings_score,0.7650,0.0000,True
5,repayment_behavior_score,0.2583,0.0000,True



Drifted features (p < 0.05): ['income_score', 'credit_history_score', 'debt_ratio_score', 'employment_score', 'savings_score', 'repayment_behavior_score']
Total drifted              : 6 / 6


/tmp/ipykernel_2187/197958402.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": dt.datetime.utcnow().isoformat()


### 3.5.3 Publish Drift Metrics and Create CloudWatch Retraining Alarm

A **CloudWatch Alarm** is created on `DriftedFeatureCount`. In production, `AlarmActions` would point to an SNS topic → Lambda → trigger the retraining pipeline. Here we read the drift count directly as the trigger condition.


In [40]:
# Block 12 - Publish drift metrics + create CloudWatch retraining alarm

for row in drift_results:
    cloudwatch.put_metric_data(
        Namespace=namespace,
        MetricData=[{"MetricName": "FeatureDrift-KS-Stat", "Value": row["ks_stat"],
                     "Unit": "None",
                     "Dimensions": [{"Name": "Project", "Value": project_name},
                                    {"Name": "Feature", "Value": row["feature"]}]}]
    )

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[{"MetricName": "DriftedFeatureCount", "Value": float(drift_count),
                 "Unit": "Count",
                 "Dimensions": [{"Name": "Project", "Value": project_name}]}]
)

RETRAIN_THRESHOLD = 2
alarm_name = f"lesson3-retrain-alarm-{account_id}"

cloudwatch.put_metric_alarm(
    AlarmName=alarm_name,
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    EvaluationPeriods=1,
    MetricName="DriftedFeatureCount",
    Namespace=namespace,
    Period=300,
    Statistic="Average",
    Threshold=float(RETRAIN_THRESHOLD),
    AlarmDescription=(
        "Fires when >= 2 features show KS-test drift (p<0.05). "
        "Production: AlarmActions = [SNS -> Lambda -> trigger pipeline]."
    ),
    Dimensions=[{"Name": "Project", "Value": project_name}]
)

RETRAINING_TRIGGERED = drift_count >= RETRAIN_THRESHOLD

print("CloudWatch alarm created:", alarm_name)
print(f"DriftedFeatureCount : {drift_count}")
print(f"Threshold           : {RETRAIN_THRESHOLD}")
print(f"RETRAIN             : {'YES' if RETRAINING_TRIGGERED else 'NO'}")

CloudWatch alarm created: lesson3-retrain-alarm-455865672536
DriftedFeatureCount : 6
Threshold           : 2
RETRAIN             : YES


## 3.6 Automated Retraining (v2)

### 3.6.1 Retrain v2 on Drifted Distribution

In [41]:
# Block 13 - Retrain v2 if drift triggered — new SageMaker Trial

if not RETRAINING_TRIGGERED:
    print("Drift below threshold — no retraining needed")
    ctx_v2 = {}
    trial_v2 = None
else:
    print("Drift threshold exceeded — training v2")
    print()

    trial_v2 = f"v2-retrained-{run_id}"
    sagemaker_client.create_trial(
        ExperimentName=experiment_name, TrialName=trial_v2,
        Tags=[{"Key": "ModelVersion", "Value": "v2"},
              {"Key": "RetrainTrigger", "Value": "drift"}]
    )
    print("Trial:", trial_v2)
    ctx_v2 = {}

    def w2_transform(ctx):
        Xs = pd.DataFrame(StandardScaler().fit_transform(df_drift[feat_cols]), columns=feat_cols)
        Xtr, Xte, ytr, yte = train_test_split(
            Xs, df_drift["loan_default_risk"], test_size=0.25, random_state=42)
        tr = Xtr.copy(); tr["loan_default_risk"] = ytr.values
        te = Xte.copy(); te["loan_default_risk"] = yte.values
        tr.to_csv(local_dir / "train_v2.csv", index=False)
        te.to_csv(local_dir / "test_v2.csv",  index=False)
        ctx["train_rows_v2"] = len(tr)
        return {}

    def w2_train(ctx):
        tr = pd.read_csv(local_dir / "train_v2.csv")
        te = pd.read_csv(local_dir / "test_v2.csv")
        m2 = RandomForestClassifier(
            n_estimators=120, max_depth=7, random_state=42, class_weight="balanced")
        m2.fit(tr[feat_cols], tr["loan_default_risk"])
        preds = m2.predict(te[feat_cols])
        ctx["accuracy_v2"] = float(accuracy_score(te["loan_default_risk"], preds))
        mp2 = local_dir / "model_v2.joblib"; joblib.dump(m2, mp2)
        with open(mp2, "rb") as f: ctx["hash_v2"] = hashlib.sha256(f.read()).hexdigest()
        print(f"    v2 accuracy: {ctx['accuracy_v2']:.4f}")
        print(classification_report(te["loan_default_risk"], preds))
        return {}

    print()
    run_stage(trial_v2, "v2-s1-transform", "v2-Stage1-Transform", w2_transform, ctx_v2,
              param_keys=["train_rows_v2"])
    run_stage(trial_v2, "v2-s2-train",     "v2-Stage2-Train",     w2_train,     ctx_v2,
              param_keys=["accuracy_v2"])

    print()
    print(f"v1 accuracy  : {ctx_v1['accuracy_v1']:.4f}")
    print(f"v2 accuracy  : {ctx_v2['accuracy_v2']:.4f}")
    print(f"Delta        : {ctx_v2['accuracy_v2'] - ctx_v1['accuracy_v1']:+.4f}")

Drift threshold exceeded — training v2

Trial: v2-retrained-20260915-060957



/tmp/ipykernel_2187/405646415.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=dt.datetime.utcnow(),


  [v2-Stage1-Transform] running...
  [v2-Stage1-Transform] completed


/tmp/ipykernel_2187/405646415.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=dt.datetime.utcnow(),
/tmp/ipykernel_2187/405646415.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=dt.datetime.utcnow(),


  [v2-Stage2-Train] running...
    v2 accuracy: 0.8200
              precision    recall  f1-score   support

           0       0.76      0.86      0.81        22
           1       0.88      0.79      0.83        28

    accuracy                           0.82        50
   macro avg       0.82      0.82      0.82        50
weighted avg       0.83      0.82      0.82        50

  [v2-Stage2-Train] completed

v1 accuracy  : 0.8560
v2 accuracy  : 0.8200
Delta        : -0.0360


/tmp/ipykernel_2187/405646415.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=dt.datetime.utcnow(),


### 3.6.2 Accuracy Gate — Register v2 or Rollback to v1

In [42]:
# Block 14 - Accuracy gate: register v2 if it passes, else rollback to v1

ACCURACY_GATE = 0.75

if not RETRAINING_TRIGGERED:
    ACTIVE_VERSION  = "v1"
    ACTIVE_ACCURACY = ctx_v1["accuracy_v1"]
    ACTIVE_MODEL    = str(local_dir / "model_v1.joblib")
    ACTIVE_KEY      = paths["dev"]["model"]
    print("v1 remains active (no retraining triggered)")
else:
    acc_v2 = ctx_v2.get("accuracy_v2", 0)
    v2_passes = acc_v2 >= ACCURACY_GATE
    print(f"Accuracy gate : {ACCURACY_GATE}")
    print(f"v2 accuracy   : {acc_v2:.4f}")
    print(f"Gate passed   : {v2_passes}\n")

    if v2_passes:
        v2_key = paths["dev"]["model"].replace(run_id, run_id + "-v2")
        s3.upload_file(str(local_dir / "model_v2.joblib"), bucket_name, v2_key)
        package_params_v2 = {
            "ModelPackageDescription": f"v2 Retrained | acc={acc_v2:.4f} | drift-triggered | run={run_id}",
            "InferenceSpecification": {
                "Containers": [{"Image": sklearn_image_uri, "ModelDataUrl": f"s3://{bucket_name}/{v2_key}"}],
                "SupportedContentTypes": ["text/csv", "application/json"],
                "SupportedResponseMIMETypes": ["text/csv", "application/json"],
            },
        }
        if group_exists:
            package_params_v2.update({
                "ModelPackageGroupName": model_group_name, "ModelApprovalStatus": "Approved",
                "CustomerMetadataProperties": {
                    "version": "v2", "accuracy": str(round(acc_v2, 4)),
                    "model_artifact": f"s3://{bucket_name}/{v2_key}",
                    "model_hash": ctx_v2["hash_v2"][:32], "retrain_trigger": "drift",
                    "drifted_features": " ".join(drifted_features),
                    "experiment": experiment_name, "trial": trial_v2, "run_id": run_id,
                },
            })
        else:
            package_params_v2["ModelPackageName"] = f"{project_name}-v2-{run_id}"
        resp_v2 = sagemaker_client.create_model_package(**package_params_v2)
        model_package_arn_v2 = resp_v2["ModelPackageArn"]
        ACTIVE_VERSION, ACTIVE_ACCURACY = "v2", acc_v2
        ACTIVE_MODEL, ACTIVE_KEY = str(local_dir / "model_v2.joblib"), v2_key
        print("v2 registered ? ARN:", model_package_arn_v2)
    else:
        print("v2 failed gate ? auto-rollback to v1")
        if group_exists:
            pkgs = sagemaker_client.list_model_packages(
                ModelPackageGroupName=model_group_name, ModelApprovalStatus="Approved",
                SortBy="CreationTime", SortOrder="Ascending", MaxResults=10,
            )["ModelPackageSummaryList"]
            rollback_arn = pkgs[0]["ModelPackageArn"] if pkgs else model_package_arn_v1
        else:
            rollback_arn = model_package_arn_v1
        detail = sagemaker_client.describe_model_package(ModelPackageName=rollback_arn)
        v1_artifact = detail["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
        print("Rolled back to ARN :", rollback_arn)
        print("v1 artifact        :", v1_artifact)
        ACTIVE_VERSION, ACTIVE_ACCURACY = "v1-rollback", ctx_v1["accuracy_v1"]
        ACTIVE_MODEL, ACTIVE_KEY = str(local_dir / "model_v1.joblib"), paths["dev"]["model"]
    print(f"\nActive version : {ACTIVE_VERSION}")


Accuracy gate : 0.75
v2 accuracy   : 0.8200
Gate passed   : True

v2 registered ? ARN: arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/4

Active version : v2


## 3.7 Environment Promotion: Dev → Staging → Prod

### 3.7.1 Copy Active Model Across Environments

In [43]:
# Block 15 - Promote dev → staging → prod via S3 copy

SOURCE_KEY = ACTIVE_KEY if RETRAINING_TRIGGERED else paths["dev"]["model"]

STAGING_GATE = 0.74
PROD_GATE    = 0.76

print(f"Promoting: {ACTIVE_VERSION}  |  accuracy: {ACTIVE_ACCURACY:.4f}")
print(f"Source   : s3://{bucket_name}/{SOURCE_KEY}")
print()

def promote_to(source_key, target_env, gate, acc):
    if acc < gate:
        print(f"  [{target_env}] BLOCKED — accuracy {acc:.4f} < gate {gate}")
        return False
    tgt = paths[target_env]["model"]
    s3.copy_object(CopySource={"Bucket": bucket_name, "Key": source_key},
                   Bucket=bucket_name, Key=tgt)
    print(f"  [{target_env}] Promoted → s3://{bucket_name}/{tgt}")
    return True

staging_ok = promote_to(SOURCE_KEY, "staging", STAGING_GATE, ACTIVE_ACCURACY)

if staging_ok:
    prod_ok = promote_to(paths["staging"]["model"], "prod", PROD_GATE, ACTIVE_ACCURACY)
else:
    print("  [prod] Skipped — staging gate not passed")
    prod_ok = False

print()
print(f"Staging: {'SUCCESS' if staging_ok else 'BLOCKED'}")
print(f"Prod   : {'SUCCESS' if prod_ok    else 'BLOCKED'}")

Promoting: v2  |  accuracy: 0.8200
Source   : s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/dev/model/model_20260915-060957-v2.joblib

  [staging] Promoted → s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/staging/model/model_20260915-060957.joblib
  [prod] Promoted → s3://lesson3-advanced-pipeline-455865672536-us-east-2/lesson3/prod/model/model_20260915-060957.joblib

Staging: SUCCESS
Prod   : SUCCESS


## 3.8 Cross-Run Comparison

### 3.8.1 Compare v1 vs v2 via SageMaker Experiments

In [44]:
# Block 16 - Compare v1 and v2 trials from SageMaker Experiments

print("SageMaker Experiment Cross-Run Comparison")
print(f"Experiment: {experiment_name}")
print()

trial_names = [trial_v1]
if RETRAINING_TRIGGERED and trial_v2:
    trial_names.append(trial_v2)

print(f"{'Trial':<32} {'Stage':<26} {'Status':<12} {'Param'}")
print("─" * 86)

for tname in trial_names:
    comps = sagemaker_client.list_trial_components(
        TrialName=tname, SortBy="CreationTime", SortOrder="Ascending"
    )["TrialComponentSummaries"]
    for comp in comps:
        detail = sagemaker_client.describe_trial_component(
            TrialComponentName=comp["TrialComponentName"]
        )
        status  = detail["Status"]["PrimaryStatus"]
        dname   = detail.get("DisplayName", comp["TrialComponentName"])
        pparams = detail.get("Parameters", {})
        acc_key = next((k for k in pparams if "accuracy" in k), None)
        pstr    = f"acc={pparams[acc_key]['NumberValue']:.4f}" if acc_key else "—"
        icon    = "✅" if status == "Completed" else "❌"
        print(f"  {icon}  {tname:<30} {dname:<26} {status:<12} {pstr}")

print()
print(f"v1 accuracy  : {ctx_v1['accuracy_v1']:.4f}")
if RETRAINING_TRIGGERED and ctx_v2:
    delta = ctx_v2['accuracy_v2'] - ctx_v1['accuracy_v1']
    print(f"v2 accuracy  : {ctx_v2['accuracy_v2']:.4f}")
    print(f"Delta        : {delta:+.4f}  ({'improvement' if delta > 0 else 'regression'})")

SageMaker Experiment Cross-Run Comparison
Experiment: lesson3-advanced-pipeline-exp

Trial                            Stage                      Status       Param
──────────────────────────────────────────────────────────────────────────────────────
  ✅  v1-baseline-20260915-060957    v1-Stage1-Extract          Completed    —
  ✅  v1-baseline-20260915-060957    v1-Stage2-Transform        Completed    —
  ✅  v1-baseline-20260915-060957    v1-Stage3-Train            Completed    acc=0.8560
  ✅  v2-retrained-20260915-060957   v2-Stage1-Transform        Completed    —
  ✅  v2-retrained-20260915-060957   v2-Stage2-Train            Completed    acc=0.8200

v1 accuracy  : 0.8560
v2 accuracy  : 0.8200
Delta        : -0.0360  (regression)


## 3.9 Monitoring

### 3.9.1 Publish Full Pipeline Metrics to CloudWatch

In [45]:
# Block 17 - Publish pipeline-level CloudWatch metrics

metric_data = [
    {"MetricName": "v1-Accuracy",         "Value": float(ctx_v1["accuracy_v1"])},
    {"MetricName": "DriftedFeatureCount", "Value": float(drift_count)},
    {"MetricName": "RetrainingTriggered", "Value": 1.0 if RETRAINING_TRIGGERED else 0.0},
    {"MetricName": "StagingPromotion",    "Value": 1.0 if staging_ok else 0.0},
    {"MetricName": "ProdPromotion",       "Value": 1.0 if prod_ok    else 0.0},
]
if RETRAINING_TRIGGERED and ctx_v2:
    metric_data.append({"MetricName": "v2-Accuracy", "Value": float(ctx_v2["accuracy_v2"])})

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[{**m, "Unit": "None",
                 "Dimensions": [{"Name": "Project", "Value": project_name}]}
                for m in metric_data]
)

print("CloudWatch metrics published — namespace:", namespace)
for m in metric_data:
    print(f"  {m['MetricName']:<28}: {m['Value']}")

CloudWatch metrics published — namespace: Lesson3/AdvancedPipeline
  v1-Accuracy                 : 0.856
  DriftedFeatureCount         : 6.0
  RetrainingTriggered         : 1.0
  StagingPromotion            : 1.0
  ProdPromotion               : 1.0
  v2-Accuracy                 : 0.82


### 3.9.2 List Model Versions in Registry

In [46]:
# Block 18 - List registered model packages

if group_exists:
    pkgs = sagemaker_client.list_model_packages(
        ModelPackageGroupName=model_group_name,
        SortBy="CreationTime", SortOrder="Ascending", MaxResults=20,
    )["ModelPackageSummaryList"]
    print(f"Model Package Group : {model_group_name}")
    print(f"Total versions      : {len(pkgs)}")
    for pkg in pkgs:
        print(" ", pkg["ModelPackageArn"], pkg["ModelApprovalStatus"])
else:
    unversioned = [model_package_arn_v1]
    if "model_package_arn_v2" in globals():
        unversioned.append(model_package_arn_v2)
    print("Unversioned Model Packages (group quota fallback):")
    for arn in unversioned:
        print(" ", arn)


Model Package Group : lesson3-loan-risk-455865672536
Total versions      : 4
  arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/1 Approved
  arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/2 Approved
  arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/3 Approved
  arn:aws:sagemaker:us-east-2:455865672536:model-package/lesson3-loan-risk-455865672536/4 Approved


### 3.9.3 List Artifacts Across All Environments

In [47]:
# Block 19 - List S3 artifacts per environment

print("Multi-Environment S3 Artifact Summary")
print()
for env in ["dev", "staging", "prod"]:
    objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=ENVS[env]).get("Contents", [])
    print(f"  [{env}] {ENVS[env]}/ — {len(objects)} artifacts")
    for obj in objects[:6]:
        print(f"    {obj['Key']}  ({obj['Size']/1024:.1f} KB)")
    if len(objects) > 6:
        print(f"    ... and {len(objects)-6} more")

Multi-Environment S3 Artifact Summary

  [dev] lesson3/dev/ — 10 artifacts
    lesson3/dev/data/processed/test.csv  (43.9 KB)
    lesson3/dev/data/processed/train.csv  (131.6 KB)
    lesson3/dev/data/raw/loan_data.csv  (174.5 KB)
    lesson3/dev/model/model_20260915-060633-v2.joblib  (431.1 KB)
    lesson3/dev/model/model_20260915-060633.joblib  (650.1 KB)
    lesson3/dev/model/model_20260915-060957-v2.joblib  (431.1 KB)
    ... and 4 more
  [staging] lesson3/staging/ — 2 artifacts
    lesson3/staging/model/model_20260915-060633.joblib  (431.1 KB)
    lesson3/staging/model/model_20260915-060957.joblib  (431.1 KB)
  [prod] lesson3/prod/ — 2 artifacts
    lesson3/prod/model/model_20260915-060633.joblib  (431.1 KB)
    lesson3/prod/model/model_20260915-060957.joblib  (431.1 KB)


## 3.10 Cleanup

### 3.10.1 Optional Cleanup

In [48]:
# Block 20 - Optional cleanup

CLEANUP = False

if CLEANUP:
    for env in ENVS:
        objs = s3.list_objects_v2(Bucket=bucket_name, Prefix=ENVS[env]).get("Contents", [])
        if objs:
            s3.delete_objects(Bucket=bucket_name, Delete={"Objects": [{"Key": o["Key"]} for o in objs]})
            print(f"Deleted {len(objs)} objects from [{env}]")
    try:
        cloudwatch.delete_alarms(AlarmNames=[alarm_name])
    except Exception:
        pass
    for tname in [t for t in [trial_v1, trial_v2 if RETRAINING_TRIGGERED else None] if t]:
        for comp in sagemaker_client.list_trial_components(TrialName=tname).get("TrialComponentSummaries", []):
            sagemaker_client.disassociate_trial_component(TrialName=tname, TrialComponentName=comp["TrialComponentName"])
            sagemaker_client.delete_trial_component(TrialComponentName=comp["TrialComponentName"])
        sagemaker_client.delete_trial(TrialName=tname)
    sagemaker_client.delete_experiment(ExperimentName=experiment_name)
    if group_exists:
        pkgs = sagemaker_client.list_model_packages(ModelPackageGroupName=model_group_name, MaxResults=100).get("ModelPackageSummaryList", [])
        for pkg in pkgs:
            sagemaker_client.delete_model_package(ModelPackageName=pkg["ModelPackageArn"])
        sagemaker_client.delete_model_package_group(ModelPackageGroupName=model_group_name)
    else:
        for arn in [model_package_arn_v1, globals().get("model_package_arn_v2")]:
            if arn:
                sagemaker_client.delete_model_package(ModelPackageName=arn)
    print("Cleanup complete")
else:
    print("Cleanup skipped ? set CLEANUP = True to delete all resources")


Cleanup skipped ? set CLEANUP = True to delete all resources


### 3.10.2 Final Checklist

This notebook covered — using **real AWS services only**:

- **AWS S3** — multi-environment prefixes (dev/staging/prod), dataset, train/test splits, model artifacts, baseline distribution, drift report, S3 copy for environment promotion
- **SageMaker Experiments** — one Experiment, two Trials (v1 and v2), each stage as a real Trial Component with status tracking and parameter recording
- **KS-test drift detection** (scipy.stats.ks_2samp) — baseline from S3, per-feature p-values, drifted features identified
- **CloudWatch** — per-feature KS statistics, drift count, retraining alarm (`put_metric_alarm`) with production-pattern description, pipeline-level metrics for both runs
- **SageMaker Model Registry** — v1 and v2 registered in the same package group; automated rollback queries the registry for the last approved version when v2 fails the gate
- **Environment promotion** — dev → staging → prod via `s3.copy_object()` with independent accuracy gates per tier


In [49]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-15 11:49:39
